In [ ]:
import os
os.environ["KERAS_BACKEND"] = "tensorflow"
import re, json, random, zlib, math, collections
import numpy as np
import pandas as pd
import tensorflow as tf
import keras
from keras import layers, ops

gpus = tf.config.list_physical_devices("GPU")
print("GPUs:", gpus)
for g in gpus:
    tf.config.experimental.set_memory_growth(g, True)
# varias GPUs (ex.: 2x T4 no Kaggle) -> MirroredStrategy; uma so -> estrategia padrao
estrategia = tf.distribute.MirroredStrategy() if len(gpus) > 1 else tf.distribute.get_strategy()
N_REPLICAS = estrategia.num_replicas_in_sync
print("replicas:", N_REPLICAS)

random.seed(42)
np.random.seed(42)

# ------------------------------ hiperparametros (ajuste aqui) ------------------------------
CSV_PATH = "/kaggle/input/datasets/wellingtonneresjose/dataset-final/dataset_final.csv.gz"     # <- confira o caminho no painel Input     # tambem aceita .csv.gz
AMOSTRA_POR_FONTE = None               # ex.: 150 para testar o pipeline rapido; None = tudo
VAL_PROMILE, TEST_PROMILE = 10, 10     # 1% validacao, 1% teste (por grupo, sem vazar traducoes)

VOCAB_SIZE = 32000                     # WordPiece compartilhado (PT + EN + codigo)
TOK_FRACAO = 0.15                      # fracao das linhas de treino usada para treinar o tokenizador
SEQ_MAX_TETO = 1024                    # corta so ~0,02% das linhas (57 entradas / 67 saidas > 1024). None = maior sequencia (2456), exige MUITA memoria com ATENCAO_FP32

BATCH_MAX = 128                        # teto de exemplos por lote
TOKENS_ENC_POR_LOTE = 4096             # orcamento de tokens por lote (controla a memoria)
TOKENS_DEC_POR_LOTE = 2560

EMBED_DIM = 256
LATENT_DIM = 1024
NUM_HEADS = 8
NUM_LAYERS = 3
DROPOUT = 0.1

EPOCHS = 40
PASSOS_POR_EPOCA = 5000                # "epoca curta": valida e salva a cada N passos. None = epoca completa
LR_PICO = 2e-4                         # (3e-4 + Post-LN + float16 deu "nan" na epoca 1)
WARMUP_STEPS = 3000
PRECISAO = "mixed_float16"             # "mixed_float16" (T4) | "mixed_bfloat16" (A100/Ampere: mais estavel) | "float32"
PRE_LN = True                          # Pre-LayerNorm: bem mais estavel que o Post-LN
ATENCAO_FP32 = True                    # atencao/softmax em float32: evita o overflow que causou os "nan" (usa mais memoria)
CKPT_A_CADA = 1000                     # salva "ultimo_modelo_unificado.keras" a cada N passos (nao perde mais trabalho)
# --- PARADA POR META (avaliada na validacao, no fim de cada epoca) ---
ACC_ALVO = 0.85                        # para quando val_accuracy >= isto ...
LOSS_ALVO = 2.0                        # ... ou val_loss <= isto (o que vier primeiro). None desliga. (880k estava em ~3.13)
RETOMAR_DE = None                      # TREINO DO ZERO. (para continuar um treino: caminho do .keras)

In [ ]:
COLS = ["id", "fonte", "tarefa", "instruction", "input", "output"]


def grupo(idv, fonte, inp):
    # linhas do mesmo "conteudo" ficam no mesmo split (evita vazamento):
    if "#" in idv:                              # linha derivada (turno/titulo): herda o grupo da conversa original
        idv = idv.split("#")[0]
        fonte = idv.rsplit("-", 1)[0]
    if fonte.startswith("aya_"):
        return "y-" + inp                      # mesma entrada = mesmo split
    if fonte in ("dialogo1", "dialogo1_pt"):
        return "d-" + idv.split("-")[-1]       # a historia em EN e a traducao em PT
    if fonte.startswith("assin"):
        return "a-" + inp                      # inferencia e similaridade do mesmo par de frases
    return idv


def divisao(chave):
    x = zlib.crc32(chave.encode("utf-8")) % 1000
    return "val" if x < VAL_PROMILE else ("test" if x < VAL_PROMILE + TEST_PROMILE else "train")


def blocos(chunksize=50000):
    # le o CSV em blocos (nao precisa caber tudo na memoria) e acrescenta a coluna `split`
    vistos = {}
    for ch in pd.read_csv(CSV_PATH, usecols=COLS, dtype=str, keep_default_na=False, chunksize=chunksize):
        if AMOSTRA_POR_FONTE:
            manter = []
            for f in ch["fonte"]:
                vistos[f] = vistos.get(f, 0) + 1
                manter.append(vistos[f] <= AMOSTRA_POR_FONTE)
            ch = ch[manter]
        ch = ch.copy()
        ch["split"] = [divisao(grupo(i, f, x)) for i, f, x in zip(ch["id"], ch["fonte"], ch["input"])]
        yield ch


meta, textos_tok, testes = [], [], []
rng = np.random.default_rng(0)
for ch in blocos():
    meta.append(ch[["id", "fonte", "tarefa", "split"]])
    s = ch[ch["split"] == "train"].sample(frac=TOK_FRACAO, random_state=int(rng.integers(1 << 30)))
    for c in ("instruction", "input", "output"):
        textos_tok += [x for x in s[c] if x.strip()]
    testes.append(ch[ch["split"] == "test"])

meta = pd.concat(meta, ignore_index=True)
test_df = pd.concat(testes, ignore_index=True)
del testes
print(meta.groupby(["fonte", "split"]).size().unstack(fill_value=0))
print(f"\n{len(meta):,} linhas | {len(textos_tok):,} textos para treinar o tokenizador")

In [ ]:
from tokenizers import Tokenizer, models, trainers, pre_tokenizers, normalizers
import glob, shutil, os

TAGS = sorted(meta["tarefa"].unique())
SPECIAL = ["[PAD]", "[UNK]", "[START]", "[END]", "[SEP]"] + [f"[{t}]" for t in TAGS]

# treino do zero (RETOMAR_DE=None): treina um tokenizador NOVO com o dataset final. Com RETOMAR_DE, usa o do modelo anexado.
achados = [] if RETOMAR_DE is None else glob.glob("/kaggle/input/models/wellingtonneresjose/model-880k/keras/default/1/wordpiece_unificado.json", recursive=True)

if achados:
    print("usando o tokenizador do modelo anexado:", achados[0])
    shutil.copy(achados[0], "wordpiece_unificado.json")
    tok = Tokenizer.from_file("wordpiece_unificado.json")
    # o vocabulário precisa conter todos os especiais e tags do dataset atual
    faltando = [t for t in SPECIAL if tok.token_to_id(t) is None]
    assert not faltando, f"tokens ausentes no vocabulário do modelo: {faltando}"
    assert tok.token_to_id("[PAD]") == 0, "PAD precisa ser o id 0"
else:
    print("nenhum tokenizador anexado: treinando um novo (só use isso se for treinar do zero)")
    tok = Tokenizer(models.WordPiece(unk_token="[UNK]", max_input_chars_per_word=400))
    tok.normalizer = normalizers.NFC()
    tok.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
    trainer = trainers.WordPieceTrainer(
        vocab_size=VOCAB_SIZE, special_tokens=SPECIAL, initial_alphabet=pre_tokenizers.ByteLevel.alphabet(),
        continuing_subword_prefix="##", show_progress=False)
    tok.train_from_iterator(textos_tok, trainer)
    tok.save("wordpiece_unificado.json")

if "textos_tok" in globals():
    del textos_tok
print("vocabulario:", tok.get_vocab_size(), "tokens | tarefas:", TAGS)

In [ ]:
PAD, UNK, START, END, SEP = (tok.token_to_id(t) for t in ["[PAD]", "[UNK]", "[START]", "[END]", "[SEP]"])
TAG_ID = {t: tok.token_to_id(f"[{t}]") for t in TAGS}
TAG_IDS = set(TAG_ID.values())
VOCAB = tok.get_vocab_size()
DTYPE = np.int16 if VOCAB <= 32767 else np.int32
ID2TOK = {i: t for t, i in tok.get_vocab().items()}


def _bytes_para_unicode():
    # mapa de bytes -> caracteres imprimiveis (o mesmo do ByteLevel do tokenizers)
    bs = list(range(ord("!"), ord("~") + 1)) + list(range(ord("¡"), ord("¬") + 1)) + list(range(ord("®"), ord("ÿ") + 1))
    cs, n = bs[:], 0
    for b in range(256):
        if b not in bs:
            bs.append(b); cs.append(256 + n); n += 1
    return dict(zip(bs, [chr(c) for c in cs]))


BYTE_DECODER = {v: k for k, v in _bytes_para_unicode().items()}


def decode_ids(ids):
    # inverso exato do tokenizador: preserva quebras de linha, indentacao e acentos
    pecas = []
    for i in ids:
        i = int(i)
        if i == END:
            break
        if i in (PAD, START, SEP, UNK) or i in TAG_IDS:
            continue
        pecas.append(ID2TOK[i])
    s = "".join(p[2:] if p.startswith("##") else p for p in pecas)
    return bytes(BYTE_DECODER[c] for c in s if c in BYTE_DECODER).decode("utf-8", errors="replace")


def encode_inputs(tags, instrs, inputs, chunk=20000):
    # entrada = [tarefa] instruction [SEP] input   (partes vazias sao omitidas)
    out = []
    for s in range(0, len(tags), chunk):
        T, I, X = tags[s:s + chunk], instrs[s:s + chunk], inputs[s:s + chunk]
        flat, owner = [], []
        for k, (a, b) in enumerate(zip(I, X)):
            for p in (a, b):
                if p.strip():
                    flat.append(p); owner.append(k)
        res = [[TAG_ID[t]] for t in T]
        feito = set()
        for k, e in zip(owner, tok.encode_batch(flat)):
            if k in feito:
                res[k].append(SEP)
            res[k].extend(e.ids)
            feito.add(k)
        out += [np.array(r, dtype=DTYPE) for r in res]
    return out


def encode_targets(texts, chunk=20000):
    dec_in, dec_out = [], []
    for s in range(0, len(texts), chunk):
        for e in tok.encode_batch(texts[s:s + chunk]):
            dec_in.append(np.array([START] + e.ids, dtype=DTYPE))
            dec_out.append(np.array(e.ids + [END], dtype=DTYPE))
    return dec_in, dec_out


# round-trip: o texto decodificado tem que ser identico ao original
for t in test_df["output"].head(300):
    assert decode_ids(tok.encode(t).ids) == __import__("unicodedata").normalize("NFC", t), "round-trip falhou"
print("round-trip ok (quebras de linha e indentacao preservadas)")

dados = {"train": ([], [], []), "val": ([], [], [])}
for ch in blocos():
    for sp in ("train", "val"):
        g = ch[ch["split"] == sp]
        if len(g) == 0:
            continue
        e = encode_inputs(g["tarefa"].tolist(), g["instruction"].tolist(), g["input"].tolist())
        di, do = encode_targets(g["output"].tolist())
        dados[sp][0].extend(e); dados[sp][1].extend(di); dados[sp][2].extend(do)

le = np.array([len(x) for x in dados["train"][0] + dados["val"][0]])
ld = np.array([len(x) for x in dados["train"][1] + dados["val"][1]])
maior_enc, maior_dec = int(le.max()), int(ld.max())
r8 = lambda n: ((int(n) + 7) // 8) * 8                     # multiplo de 8 (menos formas distintas)

# ENC_MAX = DEC_MAX = maior sequencia (entrada ou saida), para nao cortar nada
SEQ_MAX = r8(max(maior_enc, maior_dec))
if SEQ_MAX_TETO:
    SEQ_MAX = min(SEQ_MAX, r8(SEQ_MAX_TETO))
ENC_MAX = DEC_MAX = SEQ_MAX
print(f"maior entrada: {maior_enc:,} tokens | maior saida: {maior_dec:,} tokens  ->  SEQ_MAX = {SEQ_MAX:,}")
print(f"linhas acima de 512 tokens: entrada {(le > 512).sum():,} | saida {(ld > 512).sum():,}"
      f"  (de {len(le):,}); acima de 1024: {(le > 1024).sum():,} | {(ld > 1024).sum():,}")

for sp in dados:                                           # garante o teto (so age se SEQ_MAX_TETO foi usado)
    dados[sp] = tuple([a[:SEQ_MAX] for a in lista] for lista in dados[sp])

In [ ]:
MIN_LOTE = N_REPLICAS      # com varias GPUs cada replica precisa receber pelo menos 1 exemplo


class Seq2SeqBatches:
    # Lotes agrupados por comprimento + orcamento de tokens: sequencias curtas (a maioria)
    # nao sao preenchidas ate o tamanho das longas, e as muito longas vao em lotes pequenos.
    def __init__(self, enc, dec_in, dec_out, shuffle, seed=0):
        self.enc, self.dec_in, self.dec_out, self.shuffle = enc, dec_in, dec_out, shuffle
        self.rng = np.random.default_rng(seed)
        self.recent = collections.deque(maxlen=64)       # ultimos lotes entregues (para diagnosticar "nan")
        le = np.array([len(x) for x in enc]); ld = np.array([len(x) for x in dec_in])
        order = np.lexsort((self.rng.random(len(enc)), le + ld))
        self.batches, cur, max_e, max_d = [], [], 0, 0
        for i in order:
            e, d = r8(le[i]), r8(ld[i])
            ne, nd = max(max_e, e), max(max_d, d)
            if len(cur) >= MIN_LOTE and (len(cur) + 1 > BATCH_MAX or (len(cur) + 1) * ne > TOKENS_ENC_POR_LOTE
                                         or (len(cur) + 1) * nd > TOKENS_DEC_POR_LOTE):
                self.batches.append(cur)
                cur, ne, nd = [], e, d
            cur.append(i)
            max_e, max_d = ne, nd
        if cur:
            if len(cur) < MIN_LOTE and self.batches:
                self.batches[-1] += cur
            else:
                self.batches.append(cur)

    def __len__(self):
        return len(self.batches)

    def get(self, b):
        ids = self.batches[b]
        E = r8(max(len(self.enc[i]) for i in ids))
        D = r8(max(len(self.dec_in[i]) for i in ids))
        # o lote precisa ser MULTIPLO do numero de replicas: o Keras reduz o 1o lote entre as GPUs ao construir o otimizador
        # e quebra ("AddN ... shapes") se a divisao for desigual. Completamos com linhas so de PAD (nao entram na loss/acuracia).
        B = -(-len(ids) // MIN_LOTE) * MIN_LOTE
        xe = np.zeros((B, E), dtype="int32")
        xd = np.zeros((B, D), dtype="int32")
        y = np.zeros((B, D), dtype="int32")
        for r, i in enumerate(ids):
            xe[r, :len(self.enc[i])] = self.enc[i]
            xd[r, :len(self.dec_in[i])] = self.dec_in[i]
            y[r, :len(self.dec_out[i])] = self.dec_out[i]
        return {"encoder_inputs": xe, "decoder_inputs": xd}, y

    def _gen(self):
        perm = self.rng.permutation(len(self.batches)) if self.shuffle else range(len(self.batches))
        for b in perm:
            x, y = self.get(b)
            self.recent.append((int(b), x["encoder_inputs"].shape, x["decoder_inputs"].shape))
            yield x, y

    def as_dataset(self, repeat=False):
        spec = tf.TensorSpec((None, None), tf.int32)       # comprimento dinamico entre lotes
        ds = tf.data.Dataset.from_generator(
            self._gen, output_signature=({"encoder_inputs": spec, "decoder_inputs": spec}, spec))
        if repeat:
            ds = ds.repeat()                               # a cada volta a ordem dos lotes e reembaralhada
        else:
            ds = ds.apply(tf.data.experimental.assert_cardinality(len(self.batches)))
        return ds.prefetch(tf.data.AUTOTUNE)


train_gen = Seq2SeqBatches(*dados["train"], shuffle=True, seed=1)
val_gen = Seq2SeqBatches(*dados["val"], shuffle=False)
passos_epoca = PASSOS_POR_EPOCA or len(train_gen)
train_ds = train_gen.as_dataset(repeat=bool(PASSOS_POR_EPOCA))
val_ds = val_gen.as_dataset()
print(f"{len(train_gen):,} lotes por volta completa nos dados | {len(val_gen):,} lotes de validacao")
print(f"1 'epoca' = {passos_epoca:,} passos  ->  {EPOCHS} epocas = {passos_epoca * EPOCHS:,} passos")

x, y = train_gen.get(len(train_gen) // 2)
print("exemplo de lote -> encoder:", x["encoder_inputs"].shape, "| decoder:", x["decoder_inputs"].shape)
x, y = train_gen.get(0)
leg = lambda t: decode_ids([int(t)]) or ID2TOK[int(t)]           # texto legivel (e nomes dos tokens especiais)
print("alinhamento  entrada do decoder:", [leg(t) for t in x["decoder_inputs"][0][:6] if t != 0])
print("             alvo              :", [leg(t) for t in y[0][:6] if t != 0])

In [ ]:
import tensorflow as tf
import keras
from keras import layers, ops


class PositionalEmbedding(layers.Layer):
    def __init__(self, sequence_length, vocab_size, embed_dim, dropout_rate=0.1, **kwargs):
        super().__init__(**kwargs)
        self.token_embeddings = layers.Embedding(input_dim=vocab_size, output_dim=embed_dim)
        self.position_embeddings = layers.Embedding(input_dim=sequence_length, output_dim=embed_dim)
        self.sequence_length, self.vocab_size, self.embed_dim = sequence_length, vocab_size, embed_dim
        self.dropout_rate = dropout_rate
        self.dropout = layers.Dropout(dropout_rate)

    def call(self, inputs):
        positions = ops.arange(0, ops.shape(inputs)[-1], 1)
        return self.dropout(self.token_embeddings(inputs) + self.position_embeddings(positions))

    def compute_mask(self, inputs, mask=None):
        return ops.not_equal(inputs, 0)

    def get_config(self):
        config = super().get_config()
        config.update({"sequence_length": self.sequence_length, "vocab_size": self.vocab_size,
                       "embed_dim": self.embed_dim, "dropout_rate": self.dropout_rate})
        return config


def _mha(num_heads, embed_dim, dropout_rate, attn_fp32):
    # attn_fp32=True calcula a atencao (e o softmax) em float32: mais estavel, usa mais memoria
    return layers.MultiHeadAttention(num_heads=num_heads, key_dim=embed_dim // num_heads,
                                     dropout=dropout_rate, dtype="float32" if attn_fp32 else None)


class TransformerEncoder(layers.Layer):
    # pre_ln=True: normaliza ANTES de cada subcamada (estavel em float16). False = Post-LN original.
    def __init__(self, embed_dim, dense_dim, num_heads, dropout_rate=0.1, pre_ln=True, attn_fp32=False, **kwargs):
        super().__init__(**kwargs)
        self.embed_dim, self.dense_dim, self.num_heads = embed_dim, dense_dim, num_heads
        self.dropout_rate, self.pre_ln, self.attn_fp32 = dropout_rate, pre_ln, attn_fp32
        self.attention = _mha(num_heads, embed_dim, dropout_rate, attn_fp32)
        self.dense_proj = keras.Sequential([
            layers.Dense(dense_dim, activation="relu"), layers.Dropout(dropout_rate), layers.Dense(embed_dim)])
        self.layernorm_1 = layers.LayerNormalization()
        self.layernorm_2 = layers.LayerNormalization()
        self.supports_masking = True

    def call(self, inputs, mask=None):
        pad = ops.cast(mask[:, None, :], dtype="int32") if mask is not None else None
        x = inputs
        if self.pre_ln:
            h = self.layernorm_1(x)
            x = x + ops.cast(self.attention(query=h, value=h, key=h, attention_mask=pad), x.dtype)
            return x + self.dense_proj(self.layernorm_2(x))
        a = self.attention(query=x, value=x, key=x, attention_mask=pad)
        x = self.layernorm_1(x + ops.cast(a, x.dtype))
        return self.layernorm_2(x + self.dense_proj(x))

    def get_config(self):
        config = super().get_config()
        config.update({"embed_dim": self.embed_dim, "dense_dim": self.dense_dim, "num_heads": self.num_heads,
                       "dropout_rate": self.dropout_rate, "pre_ln": self.pre_ln, "attn_fp32": self.attn_fp32})
        return config


class TransformerDecoder(layers.Layer):
    def __init__(self, embed_dim, latent_dim, num_heads, dropout_rate=0.1, pre_ln=True, attn_fp32=False, **kwargs):
        super().__init__(**kwargs)
        self.embed_dim, self.latent_dim, self.num_heads = embed_dim, latent_dim, num_heads
        self.dropout_rate, self.pre_ln, self.attn_fp32 = dropout_rate, pre_ln, attn_fp32
        self.attention_1 = _mha(num_heads, embed_dim, dropout_rate, attn_fp32)
        self.attention_2 = _mha(num_heads, embed_dim, dropout_rate, attn_fp32)
        self.dense_proj = keras.Sequential([
            layers.Dense(latent_dim, activation="relu"), layers.Dropout(dropout_rate), layers.Dense(embed_dim)])
        self.layernorm_1 = layers.LayerNormalization()
        self.layernorm_2 = layers.LayerNormalization()
        self.layernorm_3 = layers.LayerNormalization()
        self.supports_masking = True

    def call(self, inputs, mask=None):
        x, enc = inputs
        x_mask, enc_mask = (None, None) if mask is None else mask
        n = ops.shape(x)[1]
        causal = ops.cast(ops.arange(n)[:, None] >= ops.arange(n), dtype="bool")[None, :, :]
        c = lambda t: ops.cast(t, x.dtype)
        if self.pre_ln:
            h = self.layernorm_1(x)
            x = x + c(self.attention_1(query=h, value=h, key=h, attention_mask=causal, query_mask=x_mask))
            h = self.layernorm_2(x)
            x = x + c(self.attention_2(query=h, value=enc, key=enc, query_mask=x_mask, key_mask=enc_mask))
            return x + self.dense_proj(self.layernorm_3(x))
        a1 = self.attention_1(query=x, value=x, key=x, attention_mask=causal, query_mask=x_mask)
        o1 = self.layernorm_1(x + c(a1))
        a2 = self.attention_2(query=o1, value=enc, key=enc, query_mask=x_mask, key_mask=enc_mask)
        o2 = self.layernorm_2(o1 + c(a2))
        return self.layernorm_3(o2 + self.dense_proj(o2))

    def compute_mask(self, inputs, mask=None):
        return mask[0] if mask is not None else None   # necessario para empilhar camadas

    def get_config(self):
        config = super().get_config()
        config.update({"embed_dim": self.embed_dim, "latent_dim": self.latent_dim, "num_heads": self.num_heads,
                       "dropout_rate": self.dropout_rate, "pre_ln": self.pre_ln, "attn_fp32": self.attn_fp32})
        return config


keras.mixed_precision.set_global_policy(PRECISAO)


def construir_modelo():
    enc_in = keras.Input(shape=(None,), dtype="int32", name="encoder_inputs")
    dec_in = keras.Input(shape=(None,), dtype="int32", name="decoder_inputs")
    kw = dict(dropout_rate=DROPOUT, pre_ln=PRE_LN, attn_fp32=ATENCAO_FP32)

    x = PositionalEmbedding(ENC_MAX, VOCAB, EMBED_DIM, DROPOUT, name="enc_embedding")(enc_in)
    for k in range(NUM_LAYERS):
        x = TransformerEncoder(EMBED_DIM, LATENT_DIM, NUM_HEADS, name=f"encoder_{k + 1}", **kw)(x)
    if PRE_LN:
        x = layers.LayerNormalization(name="enc_final_ln")(x)      # Pre-LN precisa de uma LN no fim da pilha
    enc_out = x

    y = PositionalEmbedding(DEC_MAX, VOCAB, EMBED_DIM, DROPOUT, name="dec_embedding")(dec_in)
    for k in range(NUM_LAYERS):
        y = TransformerDecoder(EMBED_DIM, LATENT_DIM, NUM_HEADS, name=f"decoder_{k + 1}", **kw)([y, enc_out])
    if PRE_LN:
        y = layers.LayerNormalization(name="dec_final_ln")(y)

    y = layers.Dropout(DROPOUT)(y)
    out = layers.Dense(VOCAB, activation="softmax", dtype="float32", name="saida")(y)
    return keras.Model({"encoder_inputs": enc_in, "decoder_inputs": dec_in}, out, name="transformer_unificado")


CUSTOM = {"PositionalEmbedding": PositionalEmbedding, "TransformerEncoder": TransformerEncoder,
          "TransformerDecoder": TransformerDecoder}

with estrategia.scope():
    if RETOMAR_DE:
        transformer = keras.models.load_model(RETOMAR_DE, custom_objects=CUSTOM, compile=False)
        print("modelo carregado de", RETOMAR_DE)
    else:
        transformer = construir_modelo()
transformer.summary()

In [ ]:
class AcuraciaMascarada(keras.metrics.Metric):
    # acertos / tokens reais (ignora o padding), todos os tokens com o mesmo peso
    def __init__(self, name="accuracy", **kwargs):
        super().__init__(name=name, **kwargs)
        self.acertos = self.add_variable(shape=(), initializer="zeros", name="acertos")
        self.total = self.add_variable(shape=(), initializer="zeros", name="total")

    def update_state(self, y_true, y_pred, sample_weight=None):
        y_true = ops.cast(y_true, "int32")
        pred = ops.cast(ops.argmax(y_pred, axis=-1), "int32")
        real = ops.cast(ops.not_equal(y_true, 0), "float32")
        self.acertos.assign_add(ops.sum(ops.cast(ops.equal(pred, y_true), "float32") * real))
        self.total.assign_add(ops.sum(real))

    def result(self):
        return self.acertos / ops.maximum(self.total, 1.0)

    def reset_state(self):
        self.acertos.assign(0.0)
        self.total.assign(0.0)


total_steps = passos_epoca * EPOCHS
lr = keras.optimizers.schedules.CosineDecay(
    initial_learning_rate=0.0, decay_steps=max(1, total_steps - WARMUP_STEPS),
    warmup_target=LR_PICO, warmup_steps=WARMUP_STEPS, alpha=0.05)

class LossScaleSeguro(keras.optimizers.LossScaleOptimizer):
    # CORRECAO de um defeito do Keras 3.15 com varias GPUs: o LossScaleOptimizer padrao confere se os gradientes
    # sao finitos ANTES do all-reduce e usa so a replica 0. Se o overflow de float16 ocorre so na replica 1,
    # o "inf" entra na soma e corrompe os pesos (era a causa dos "nan"). Aqui a flag de finitude e combinada
    # entre as replicas: o passo so e aplicado se TODAS estiverem finitas; senao e pulado e a escala cai.
    def check_finite(self, grads):
        finite = super().check_finite(grads)
        ctx = tf.distribute.get_replica_context()
        if ctx is not None and ctx.num_replicas_in_sync > 1:
            n = ctx.all_reduce(tf.distribute.ReduceOp.SUM, tf.cast(finite, tf.float32))
            return tf.equal(n, float(ctx.num_replicas_in_sync))
        return finite


with estrategia.scope():
    otimizador = keras.optimizers.AdamW(learning_rate=lr, weight_decay=1e-2, beta_2=0.98, clipnorm=1.0)
    if PRECISAO == "mixed_float16":
        otimizador = LossScaleSeguro(otimizador)     # bfloat16 e float32 nao usam escala de perda
    transformer.compile(
        optimizer=otimizador,
        loss=keras.losses.SparseCategoricalCrossentropy(ignore_class=0),
        metrics=[AcuraciaMascarada()],
        jit_compile=False,       # com formas variaveis o XLA recompila a cada lote novo
    )


def pesos_finitos(modelo):
    return all(np.isfinite(w).all() for w in modelo.get_weights())


class NaNGuard(keras.callbacks.Callback):
    # Em vez de matar o treino na 1a loss "nan" (pode ser so um lote com overflow, que o otimizador pula),
    # confere se os PESOS estao ok. Se estiverem corrompidos, para e mostra os ultimos lotes.
    def __init__(self, a_cada=50):
        super().__init__()
        self.a_cada, self.n, self.avisou = a_cada, 0, False

    def on_epoch_begin(self, epoch, logs=None):
        self.avisou = False

    def on_train_batch_end(self, batch, logs=None):
        self.n += 1
        loss = (logs or {}).get("loss", 0.0)
        if not np.isfinite(loss) and not self.avisou:
            self.avisou = True
            print(f"\n[NaNGuard] loss nao-finita no passo {self.n} (lote {batch} da epoca). Pesos ok? {pesos_finitos(self.model)}")
            print("[NaNGuard] ultimos lotes (indice, encoder, decoder):", list(train_gen.recent)[-8:], flush=True)
        if self.n % self.a_cada == 0 and not pesos_finitos(self.model):
            print(f"\n[NaNGuard] PESOS corrompidos no passo {self.n}: parando o treino.", flush=True)
            self.model.stop_training = True

    def on_epoch_end(self, epoch, logs=None):
        v = (logs or {}).get("val_loss")
        if v is not None and not np.isfinite(v):
            print("[NaNGuard] val_loss nao-finita: parando o treino.", flush=True)
            self.model.stop_training = True


class SalvarSeguro(keras.callbacks.Callback):
    # salva a cada N passos, mas so se os pesos estiverem finitos (nunca grava um modelo corrompido)
    def __init__(self, caminho, a_cada):
        super().__init__()
        self.caminho, self.a_cada, self.n = caminho, a_cada, 0

    def on_train_batch_end(self, batch, logs=None):
        self.n += 1
        if self.n % self.a_cada == 0 and pesos_finitos(self.model):
            self.model.save(self.caminho)


class SalvarMelhor(keras.callbacks.Callback):
    # substitui o ModelCheckpoint(save_best_only): aquele pode gravar um modelo com "nan" como "melhor"
    def __init__(self, caminho):
        super().__init__()
        self.caminho, self.melhor = caminho, np.inf

    def on_epoch_end(self, epoch, logs=None):
        v = (logs or {}).get("val_loss")
        if v is not None and np.isfinite(v) and v < self.melhor and pesos_finitos(self.model):
            self.melhor = v
            self.model.save(self.caminho)
            print(f"[SalvarMelhor] val_loss {v:.4f} -> salvo em {self.caminho}", flush=True)


class ParadaPorMeta(keras.callbacks.Callback):
    # encerra o treino quando a validacao atinge a meta de acuracia ou de loss (checa no fim de cada epoca)
    def on_epoch_end(self, epoch, logs=None):
        logs = logs or {}
        acc, loss = logs.get("val_accuracy"), logs.get("val_loss")
        motivo = None
        if ACC_ALVO is not None and acc is not None and acc >= ACC_ALVO:
            motivo = f"val_accuracy {acc:.4f} >= {ACC_ALVO}"
        elif LOSS_ALVO is not None and loss is not None and np.isfinite(loss) and loss <= LOSS_ALVO:
            motivo = f"val_loss {loss:.4f} <= {LOSS_ALVO}"
        if motivo:
            print(f"\n[ParadaPorMeta] meta atingida ({motivo}) na epoca {epoch + 1}: encerrando o treino.", flush=True)
            self.model.stop_training = True


callbacks = [
    NaNGuard(),
    SalvarSeguro("ultimo_modelo_unificado.keras", CKPT_A_CADA),
    SalvarMelhor("melhor_modelo_unificado.keras"),
    ParadaPorMeta(),
    keras.callbacks.EarlyStopping(monitor="val_loss", patience=4, restore_best_weights=True),
]
transformer.fit(train_ds, epochs=EPOCHS, steps_per_epoch=passos_epoca if PASSOS_POR_EPOCA else None,
                validation_data=val_ds, callbacks=callbacks, shuffle=False, verbose=2)

In [ ]:
transformer.save("modelo_unificado.keras")
config = dict(TAGS=TAGS, SEQ_MAX=SEQ_MAX, VOCAB_SIZE=VOCAB, EMBED_DIM=EMBED_DIM, LATENT_DIM=LATENT_DIM,
              NUM_HEADS=NUM_HEADS, NUM_LAYERS=NUM_LAYERS)
with open("config_unificado.json", "w") as f:
    json.dump(config, f)
print("Salvos: modelo_unificado.keras, wordpiece_unificado.json, config_unificado.json")

In [ ]:
MODEL = transformer

@tf.function(input_signature=[tf.TensorSpec([None, None], tf.int32), tf.TensorSpec([None, None], tf.int32)],
             autograph=False)
def _forward(e, d):
    return MODEL({"encoder_inputs": e, "decoder_inputs": d}, training=False)


def _pad(seqs):
    L = max(len(s) for s in seqs)
    a = np.zeros((len(seqs), L), dtype="int32")
    for r, s in enumerate(seqs):
        a[r, :len(s)] = s
    return a


def gerar_lote(prompts, max_new=256, tamanho_lote=16):
    # prompts = [(tarefa, instruction, input), ...]  -> lista de textos gerados (decodificacao gulosa)
    resultados = []
    max_new = min(max_new, SEQ_MAX - 1)
    for ini in range(0, len(prompts), tamanho_lote):
        chunk = prompts[ini:ini + tamanho_lote]
        tags, ins, inp = zip(*chunk)
        enc = _pad([a[:SEQ_MAX] for a in encode_inputs(list(tags), list(ins), list(inp))])
        dec = np.full((len(chunk), 1), START, dtype="int32")
        fim = np.zeros(len(chunk), dtype=bool)
        for _ in range(max_new):
            probs = _forward(enc, dec).numpy()[:, -1, :]
            prox = probs.argmax(-1).astype("int32")
            prox[fim] = PAD
            dec = np.concatenate([dec, prox[:, None]], axis=1)
            fim |= (prox == END)
            if fim.all():
                break
        resultados += [decode_ids(row[1:]) for row in dec]
    return resultados

In [ ]:
# Acerto exato nas tarefas de saida curta (assin1/assin2: classe e nota)
for f in ["assin1", "assin2"]:
    g = test_df[test_df["fonte"] == f].head(200)
    if len(g) == 0:
        continue
    pred = gerar_lote(list(zip(g["tarefa"], g["instruction"], g["input"])), max_new=8)
    for t in sorted(g["tarefa"].unique()):
        m = (g["tarefa"] == t).values
        acc = np.mean([p.strip() == r.strip() for p, r, k in zip(pred, g["output"], m) if k])
        print(f"{f:8s} {t:24s} acerto exato: {acc:.3f}  (n={int(m.sum())})")

# Um exemplo de cada fonte (primeiros caracteres)
for f, g in test_df.groupby("fonte"):
    r = g.iloc[0]
    saida = gerar_lote([(r["tarefa"], r["instruction"], r["input"])], max_new=120)[0]
    print("=" * 70)
    print(f"[{f}] ENTRADA : {' | '.join(p for p in (r['instruction'], r['input']) if p)[:200]!r}")
    print(f"        MODELO  : {saida[:300]!r}")
    print(f"        ESPERADO: {r['output'][:300]!r}")